# Room noise: raw vs denoised, and what is firing barge-in

You are on earphones, so the agent's voice is not reaching the mic — and the log
agrees. Measured across `kb-20260819-152919`, the raw mic noise floor while the
agent is speaking is **−32.4 dBFS**, and while it is silent, **−31.4 dBFS**. One
decibel. Nothing about the agent talking makes your mic louder: it is not echo,
and it is not AGC pumping.

What is actually happening is narrower, and this notebook is built to let you
*hear* it:

> Your room noise crosses the VAD threshold in short bursts all the time. When the
> agent is silent those bursts are harmless — the endpointer needs words before it
> will end a turn, and noise has none. When the agent is **speaking**, the very
> same burst hits `agent_app.py:720`, which in `echo_mode="off"` interrupts on the
> onset event alone, with no confirmation of any kind. Same noise, no new noise —
> just one code path that acts on it and one that doesn't.

So the cells below play you three things, raw and GTCRN-denoised side by side:

1. **the bursts that actually cancelled a reply** — the exact audio, ±1.5 s of context
2. **the room floor on its own** — every stretch the gate called speech but that produced no words, stitched into one clip
3. **your real speech** — the same treatment, as the A/B reference

`.mic.wav` is stored raw ("exactly as the browser sent it", per `measure.py`), so
the denoised side is produced here by running the live GTCRN through the same
1280-sample chunks the pipeline uses. What you hear is what the gate heard.

## 1 · Setup

In [1]:
import asyncio, json, sys, time
from pathlib import Path

import numpy as np
import soundfile as sf
from IPython.display import Audio, HTML, display

REC = Path("recordings")
assert REC.is_dir() and Path("server").is_dir(), (
    f"run this notebook from demo-v3/, not {Path.cwd()} — it imports the live denoiser")

sys.path.insert(0, str(Path.cwd()))
from server.speech.denoise import make_denoiser

SR         = 16_000
CHUNK      = 1280          # 80 ms — the live pipeline's chunk
VAD_BAR    = 0.6           # SpeechGate threshold from the session header
AGENT_HOLD = 200.0         # ms after a ref chunk that the agent is still audible

_den = make_denoiser("gtcrn", model_path="models/gtcrn_simple.onnx")
await _den.start()

AssertionError: run this notebook from demo-v3/, not /workspace/speech_demo/demo-v3/notebooks — it imports the live denoiser

## 2 · Load a session

`denoise()` streams the raw wav through GTCRN in 80 ms chunks with the model's
state carried across them — the same way the live session does it. Batching the
whole file through in one call would give the model lookahead it never has at
runtime, and would flatter it.

In [ ]:
def denoise(y: np.ndarray) -> np.ndarray:
    """Run GTCRN over `y` the way the live pipeline does: 80 ms at a time."""
    if len(y) == 0:
        return y
    _den.reset()
    return np.concatenate([_den.process(y[i:i + CHUNK]) for i in range(0, len(y), CHUNK)])


def load(session: str) -> dict:
    """Load one recording: raw audio, denoised audio, and the per-chunk telemetry."""
    stem = session.replace(".mic.wav", "").replace(".jsonl", "")
    y, sr = sf.read(str(REC / f"{stem}.mic.wav"), dtype="float32")
    assert sr == SR, sr

    rows  = [json.loads(l) for l in (REC / f"{stem}.jsonl").read_text().splitlines() if l.strip()]
    chunks = [r for r in rows if r["type"] == "chunk"]
    refs   = np.array(sorted(r["t_ms"] for r in rows if r["type"] == "ref"))
    head   = next(r for r in rows if r["type"] == "header")

    t = np.array([c["t_ms"] for c in chunks])
    # The agent is audible if a ref (playback) chunk went out in the last 200 ms.
    i  = np.searchsorted(refs, t)
    ag = (i > 0) & ((t - refs[np.maximum(i - 1, 0)]) <= AGENT_HOLD) if len(refs) else np.zeros(len(t), bool)

    return {
        "stem":  stem,
        "raw":   y,
        "clean": denoise(y),
        "t":     t,
        "vad":   np.array([c["vad"] for c in chunks]),
        "dbfs":  np.array([c["dbfs"] for c in chunks]),
        "clean_dbfs": np.array([c.get("extra", {}).get("dbfs_clean", np.nan) for c in chunks]),
        "speech": np.array([bool(c.get("extra", {}).get("is_speech")) for c in chunks]),
        "agent": ag,
        "meta":  head["meta"],
    }


S = load("kb-20260819-152919")
print(f"{S['stem']}  ·  {len(S['raw'])/SR:.1f}s  ·  echo_mode={S['meta']['echo_mode']}  "
      f"·  vad threshold={S['meta']['threshold']} release={S['meta']['release_threshold']}")

## 3 · The whole session, raw vs denoised

Boost with `gain_db` — these recordings sit around −27 dBFS and are hard to judge
at unity on laptop speakers. The gain is applied to both sides equally, so the
comparison stays honest.

In [ ]:
def player(y, gain_db: float = 12.0, label: str = ""):
    g = 10 ** (gain_db / 20)
    a = np.clip(y * g, -1, 1)
    if label:
        display(HTML(f"<div style='font:12px ui-monospace,Menlo,monospace;opacity:.7;"
                     f"margin:6px 0 2px'>{label}</div>"))
    display(Audio(a, rate=SR))


def ab(raw, clean, gain_db: float = 12.0, note: str = ""):
    """Raw over denoised, same gain, so you can flip between them."""
    if note:
        display(HTML(f"<div style='font:13px ui-monospace,Menlo,monospace;margin:10px 0 0'>{note}</div>"))
    player(raw,   gain_db, "RAW — what the browser sent")
    player(clean, gain_db, "DENOISED — what the VAD and the recogniser heard")


ab(S["raw"], S["clean"], note=f"<b>{S['stem']}</b> — full session, {len(S['raw'])/SR:.0f}s")

## 4 · The bursts that cancelled a reply

In `echo_mode="off"` barge-in fires on a speech **onset** that lands while the
agent is audible. That is a two-line condition, and it is reproducible offline
from the telemetry alone — no log parsing needed. `false_triggers()` finds exactly
the events that would have called `interrupt("barge-in")`.

Each one plays with 1.5 s of lead-in so you can hear what the room was doing
before the gate opened.

In [ ]:
def speech_runs(mask, t):
    """Contiguous True runs in `mask` as (start_ms, duration_ms)."""
    out, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j + 1 < len(mask) and mask[j + 1]:
                j += 1
            out.append((t[i], t[j] - t[i] + 80.0))
            i = j + 1
        else:
            i += 1
    return out


def false_triggers(S):
    """Speech onsets while the agent was audible — i.e. every barge-in fired."""
    return speech_runs(S["speech"] & S["agent"], S["t"])


def hear(S, t_ms, dur_ms, pad_ms=1500.0, gain_db=12.0):
    a = max(0, int((t_ms - pad_ms) / 1000 * SR))
    b = min(len(S["raw"]), int((t_ms + dur_ms + pad_ms) / 1000 * SR))
    sel = (S["t"] >= t_ms - pad_ms) & (S["t"] <= t_ms + dur_ms + pad_ms)

    display(HTML(
        f"<div style='font:13px ui-monospace,Menlo,monospace;margin:14px 0 0'>"
        f"<b>t = {t_ms/1000:.1f}s</b> · burst lasted <b>{dur_ms:.0f}ms</b> · "
        f"raw peak {S['dbfs'][sel].max():.1f} dBFS · "
        f"denoised peak {np.nanmax(S['clean_dbfs'][sel]):.1f} dBFS · "
        f"VAD peak {S['vad'][sel].max():.2f} (bar {VAD_BAR})</div>"))
    player(S["raw"][a:b],   gain_db, "RAW")
    player(S["clean"][a:b], gain_db, "DENOISED")


trig = false_triggers(S)
print(f"{len(trig)} onsets fired barge-in in this session: "
      f"{[int(d) for _, d in trig]} ms\n")
for t_ms, dur in trig:
    hear(S, t_ms, dur)

## 5 · The room floor on its own

Every stretch the gate called speech in this session, stitched end to end with a
short gap between them, ordered quietest-VAD first. The early ones are the room.
Somewhere in the middle they turn into you. Listening to it this way makes the
boundary obvious in a way the per-event clips above do not.

In [ ]:
def floor_reel(S, limit=25, gap_ms=250, gain_db=12.0, only_agent=False):
    """Stitch the gate's speech stretches into one clip, quietest VAD first."""
    mask = S["speech"] & (S["agent"] if only_agent else True)
    runs = speech_runs(mask, S["t"])

    scored = []
    for t_ms, dur in runs:
        sel = (S["t"] >= t_ms) & (S["t"] < t_ms + dur)
        scored.append((S["vad"][sel].max(), t_ms, dur))
    scored.sort()

    gap = np.zeros(int(gap_ms / 1000 * SR), dtype=np.float32)
    parts, rows = [], []
    for v, t_ms, dur in scored[:limit]:
        a, b = int(t_ms / 1000 * SR), int((t_ms + dur) / 1000 * SR)
        parts += [S["raw"][a:b], gap]
        rows.append((v, t_ms, dur))

    print(f"{len(runs)} speech stretches; playing the {len(rows)} with the lowest VAD peak\n")
    print(f"{'vad peak':>9} {'at':>9} {'ms':>6}")
    for v, t_ms, dur in rows:
        print(f"{v:9.2f} {t_ms/1000:8.1f}s {dur:6.0f}")
    player(np.concatenate(parts), gain_db, "RAW — quietest-first reel")
    player(denoise(np.concatenate(parts)), gain_db, "DENOISED — same reel")


floor_reel(S)

## 6 · Your real speech, for reference

Same treatment on the loudest stretches. This is the other end of the scale —
what the gate sees when you are genuinely talking.

In [ ]:
def speech_reel(S, limit=6, gap_ms=250, gain_db=12.0):
    runs = speech_runs(S["speech"], S["t"])
    scored = sorted(
        ((S["vad"][(S["t"] >= t) & (S["t"] < t + d)].max(), t, d) for t, d in runs),
        reverse=True)
    gap = np.zeros(int(gap_ms / 1000 * SR), dtype=np.float32)
    parts = []
    for v, t_ms, dur in scored[:limit]:
        a, b = int(t_ms / 1000 * SR), int((t_ms + dur) / 1000 * SR)
        parts += [S["raw"][a:b], gap]
        print(f"vad {v:.2f}  at {t_ms/1000:6.1f}s  {dur:5.0f}ms")
    player(np.concatenate(parts), gain_db, "RAW — loudest-first reel")
    player(denoise(np.concatenate(parts)), gain_db, "DENOISED — same reel")


speech_reel(S)

## 7 · Where the two separate

The clips above are the argument; this is the same thing as numbers. `dbfs` alone
does **not** separate your room from your voice — GTCRN's output level and the VAD
peak do.

In [ ]:
def separation(S):
    noise  = S["speech"] & S["agent"]        # fired barge-in, produced no words
    voice  = S["speech"] & ~S["agent"]       # the stretches that became turns
    quiet  = ~S["speech"]

    rows = []
    for lbl, sel in [("room (fired barge-in)", noise), ("your voice", voice), ("gate closed", quiet)]:
        if not sel.any():
            continue
        rows.append((lbl, sel.sum(),
                     np.median(S["dbfs"][sel]), np.nanmedian(S["clean_dbfs"][sel]),
                     np.median(S["vad"][sel]), np.percentile(S["vad"][sel], 95)))

    print(f"{'':24} {'n':>5} {'raw dBFS':>9} {'clean dBFS':>11} {'vad med':>8} {'vad p95':>8}")
    for lbl, n, raw, cln, vm, vp in rows:
        print(f"{lbl:24} {n:5} {raw:9.1f} {cln:11.1f} {vm:8.2f} {vp:8.2f}")


separation(S)

### Across every RAG session

One session could be a fluke. This runs the same detector over all of them and
collects the duration of every burst that fired barge-in.

In [ ]:
durations = []
for jsonl in sorted(REC.glob("kb-*.jsonl")):
    Si = load(jsonl.stem)
    if len(Si["raw"]) == 0 or not len(Si["t"]):
        print(f"  {jsonl.stem:26} empty recording, nothing to score")
        continue
    trig = false_triggers(Si)
    if trig:
        durations += [d for _, d in trig]
    print(f"  {jsonl.stem:26} {len(trig):2} onsets  {sorted(int(d) for _, d in trig)}")

durations = np.array(durations)
print(f"\n{len(durations)} onsets fired barge-in across all sessions."
      f"  longest: {durations.max():.0f}ms\n")
for cut in (160, 240, 320, 400, 500, 640):
    print(f"  require >= {cut:4}ms sustained  ->  {(durations >= cut).sum():3} survive, "
          f"{(durations < cut).sum():3} suppressed")

## 8 · What this says about the fix

Every burst that cancelled a reply, in every RAG session on disk, is **≤400 ms**.
None reach 500. Not one of them produced a word — the pipeline itself said so
twice, logging `abandoned utterance: … no words` a few seconds after the
interruption.

That points at a sustain requirement rather than a threshold change:

- **raising the VAD bar** would work here (your room peaks ~0.68, your voice hits
  0.98) but it is the same knob the endpointer uses, and moving it changes turn
  detection too
- **requiring ~500 ms of sustained speech before barge-in only** leaves the
  endpointer untouched and suppresses all 19 bursts

The cost is that a genuine interruption is acknowledged ~500 ms later than it is
now. **Caveat worth stating plainly:** there are no true barge-ins anywhere in
these recordings, so this data can measure what a sustain rule suppresses but
*cannot* measure what it would wrongly delay. 500 ms is comfortably shorter than
a real interrupting phrase, but that is reasoning, not evidence.

`--echo-mode guard` is the zero-code version: barge-in fires on recognised words
instead of onsets, and none of these 19 bursts ever produced one.

## 9 · Cleanup

In [ ]:
await _den.stop()
print("denoiser released")